# managing null values

In [128]:
# read the csv
import pandas as pd
movies_df = pd.read_csv("./data/movies.csv")
movies_df

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN
1,Masters of the Universe: Revelation,(2021– ),"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN
2,The Walking Dead,(2010–2022),"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN
3,Rick and Morty,(2013– ),"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN
4,Army of Thieves,(2021),"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
9994,The Imperfects,(2021– ),"\nAdventure, Drama, Fantasy",NaN,\nAdd a Plot\n,\n \n Stars:\nMorgan Taylor Camp...,NaN,NaN,NaN
9995,Arcane,(2021– ),"\nAnimation, Action, Adventure",NaN,\nAdd a Plot\n,\n,NaN,NaN,NaN
9996,Heart of Invictus,(2022– ),"\nDocumentary, Sport",NaN,\nAdd a Plot\n,\n Director:\nOrlando von Einsiedel\n| \n ...,NaN,NaN,NaN
9997,The Imperfects,(2021– ),"\nAdventure, Drama, Fantasy",NaN,\nAdd a Plot\n,\n Director:\nJovanka Vuckovic\n| \n Sta...,NaN,NaN,NaN


In [129]:
# none
import math
print(f"="*60)
print("None values")
print(f"="*60)
value_01 = None
print(value_01 is None)
print(type(value_01))


# nan
print(f"="*60)
print("nan")
print(f"="*60)
value_02 = float('nan')
print(math.isnan(value_02))
print(value_02 == value_02)

nums = [1, 2.2, 3, float('nan'), 4, 5]
for i in nums:
    if isinstance(i, float):
        print(f"i: {i} -> is instance of float")
        print(f"i: {i} -> is nan?: {math.isnan(i)}")
        
print(f"="*60)
print("null in json")
print(f"="*60)
import json
body = json.loads('{"name": "Smith", "age": null}')
print(f"age: {body["age"]}")
print(f"age is null? -> {body["age"] is None}")

None values
True
<class 'NoneType'>
nan
True
False
i: 2.2 -> is instance of float
i: 2.2 -> is nan?: False
i: nan -> is instance of float
i: nan -> is nan?: True
null in json
age: None
age is null? -> True


In [130]:
print("peer column name")
print(movies_df.isnull().sum())

print("shape")
print(movies_df.shape)

movies_df.info()

peer column name
MOVIES         0
YEAR         644
GENRE         80
RATING      1820
ONE-LINE       0
STARS          0
VOTES       1820
RunTime     2958
Gross       9539
dtype: int64
shape
(9999, 9)
<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MOVIES    9999 non-null   str    
 1   YEAR      9355 non-null   str    
 2   GENRE     9919 non-null   str    
 3   RATING    8179 non-null   float64
 4   ONE-LINE  9999 non-null   str    
 5   STARS     9999 non-null   str    
 6   VOTES     8179 non-null   str    
 7   RunTime   7041 non-null   float64
 8   Gross     460 non-null    str    
dtypes: float64(2), str(7)
memory usage: 703.2 KB


## remove nulls 

In [131]:
movies_df_droped = movies_df.dropna()
print(movies_df_filled.describe())
movies_df_droped.shape

            RATING      RunTime
count  8179.000000  9999.000000
mean      6.921176    68.688539
std       1.220232    39.655700
min       1.100000     1.000000
25%       6.200000    45.000000
50%       7.100000    68.688539
75%       7.800000    86.000000
max       9.900000   853.000000


(460, 9)

## replace using median && mean

In [132]:
import math
import statistics
    
values = []
for rows in movies_df[["RunTime"]].itertuples():
    runtime_value = rows.RunTime
    if runtime_value is not None and not (isinstance(runtime_value, float) and math.isnan(runtime_value)):
        values.append(runtime_value)

runtime_median = statistics.median(values)
runtime_mean = statistics.mean(values)
runtime_zero = 0
print(f"runtime_median {runtime_median}")
print(f"runtime_mean {runtime_mean}")
print(f"runtime_zero {runtime_zero}")


movies_df_filled = movies_df.fillna({"RunTime": runtime_median})
print(movies_df_filled.describe())

movies_df_filled = movies_df.fillna({"RunTime": runtime_mean})
print(movies_df_filled.describe())


runtime_median 60.0
runtime_mean 68.68853855986366
runtime_zero 0
            RATING      RunTime
count  8179.000000  9999.000000
mean      6.921176    66.118212
std       1.220232    39.853505
min       1.100000     1.000000
25%       6.200000    45.000000
50%       7.100000    60.000000
75%       7.800000    86.000000
max       9.900000   853.000000
            RATING      RunTime
count  8179.000000  9999.000000
mean      6.921176    68.688539
std       1.220232    39.655700
min       1.100000     1.000000
25%       6.200000    45.000000
50%       7.100000    68.688539
75%       7.800000    86.000000
max       9.900000   853.000000


In [134]:
# IMPUTACIÓN: FORWARD FILL (usa valor anterior)

def forward_fill(df, column_name):
    values = df[column_name]
    
    for i in range(1, len(values)):
        value = values[i]
        if value is not None and not (isinstance(value, float) and math.isnan(value)):
            values[i] = values[i-1]
    
    df[column_name] = values
    return df

movies_df_filled = forward_fill(movies_df, "RunTime")
print(movies_df_filled.describe())
        

            RATING  RunTime
count  8179.000000      4.0
mean      6.921176    121.0
std       1.220232      0.0
min       1.100000    121.0
25%       6.200000    121.0
50%       7.100000    121.0
75%       7.800000    121.0
max       9.900000    121.0
